# データの特徴

In [1]:
!pip install optuna optuna-integration

# import lightgbm as lgb
import optuna.integration.lightgbm as lgb
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

from sklearn import metrics
import matplotlib.pyplot as plt

import optuna

/Users/shigeyuki-t/anaconda3/lib/python3.7/site-packages/dask/config.py:168: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  data = yaml.load(f.read()) or {}
/Users/shigeyuki-t/anaconda3/lib/python3.7/site-packages/distributed/config.py:20: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  defaults = yaml.load(f)


In [2]:
file_path = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/05.csv'
dataset = pd.read_csv(file_path)
dataset.head(3)

FileNotFoundError: [Errno 2] File b'/Users/shigeyuki-t/Desktop/FirstExperiment/\xe5\x88\x86\xe6\x9e\x90\xe7\x94\xa8/05.csv' does not exist: b'/Users/shigeyuki-t/Desktop/FirstExperiment/\xe5\x88\x86\xe6\x9e\x90\xe7\x94\xa8/05.csv'

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
# フォルダ・ファイルの作成
dataset = dataset[(dataset['sensor_id'] != 'koizumi.riku.kv8@naist.jp') & (dataset['sensor_id'] != 'fukuryu115@gmail.com')]
# print(dataset['sensor_id'].value_counts())

unique_sensor_ids = dataset['sensor_id'].unique()
base_path = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析/個人/'

for sensor_id in unique_sensor_ids:
    # フォルダ作成
    folder_name = f'{sensor_id}'
    folder_path = os.path.join(base_path, folder_name)
    
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f'created:b {folder_path}')
    else:
        print(f'already exists: {folder_path}')
    
    csv_name =  f'{sensor_id}' + '.csv'
    csv_path = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析/個人/' + f'{sensor_id}' + '/' 
    mail_dataset = dataset[dataset['sensor_id'] == f'{sensor_id}']
    csv_create_location = os.path.join(csv_path, csv_name)
    mail_dataset.to_csv(csv_create_location, index = True)
    
    if not os.path.exists(csv_create_location):
        os.makedirs(csv_create_location)
        print(f'created:b {csv_create_location}')
    else:
        print(f'already exists: {csv_create_location}')

# データ成形

In [ ]:
# for sensor_id in unique_sensor_ids:
#     # フォルダ作成
#     folder_name = f'{sensor_id}'
#     folder_path = os.path.join(base_path, folder_name)



# データの読み込み
data_clean = pd.read_csv('/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/datacheck.csv')

data_clean = data_clean[data_clean['taskNum'] < 151]

# True = 1, False = 0, レスポンス以外のときの値が-1
data_clean.loc[data_clean['event'] != 'response', 'isAnswerCorrect'] = np.nan
data_clean['isAnswerCorrect'].fillna(-1, inplace=True)

# data_timeをdatetimeに変換してソート
data_clean['data_time'] = pd.to_datetime(data_clean['data_time'])
data_clean = data_clean.sort_values(by='data_time')

# data_time の差分を秒単位で計算し、response_time に代入
data_clean.loc[data_clean['taskNum'] == 1, 'response_time'] = data_clean['data_time'].diff().dt.total_seconds().shift(-1)

# カテゴリ変数をダミー変数に変換
data_clean = pd.get_dummies(data_clean, columns=['event'])

# タップポジションを画面比に応じた値に変更
data_clean['ratio_tap_position_x'] = data_clean['tap_position_x'] / data_clean['screen_width']
data_clean['ratio_tap_position_y'] = data_clean['tap_position_y'] / data_clean['screen_height']

# スクロールの処理
data_clean['scroll_group'] = 0
group_number = 1
# 前の行の値を保存する変数
previous_value = data_clean.iloc[0]['scroll_time']
data_clean.at[0, 'scroll_group'] = group_number
# グループ分けのループ
for index in range(1, len(data_clean)):
    current_value = data_clean.iloc[index]['scroll_time']
    if current_value >= previous_value:
        data_clean.at[index, 'scroll_group'] = group_number
    else:
        group_number += 1
        data_clean.at[index, 'scroll_group'] = group_number
    previous_value = current_value

# scroll_timeが複数のtaskNumに跨って継続しているときの処理
max_task_num = data_clean['taskNum'].max()
for task_num in range(max_task_num):
    current_task_group = data_clean[data_clean['taskNum']==task_num]
    next_task_group = data_clean[data_clean['taskNum']==task_num+1]
    # np.intersect1dは複数配列の共通項を取り出す
    common_groups = np.intersect1d(current_task_group['scroll_group'].unique(),next_task_group['scroll_group'].unique())
    
    for group in common_groups:
        last_scroll_time_current = current_task_group[current_task_group['scroll_group'] == group]['scroll_time'].values[-1]
        data_clean.loc[(data_clean['taskNum'] == task_num + 1)&(data_clean['scroll_group'] == group),'scroll_time'] -= last_scroll_time_current

data_clean.head(20)

In [ ]:
# taskNumが同じ行を１行にまとめる
# 集計方法を指定
aggregation_functions = {
    "choice_left": "max", 
    "choice_right": "max",
    "isAnswerCorrect": "max",
    "response_time": "max",
#     "response_time_ave": "mean",
    "screen_height": "max",
    "screen_width": "max",
    "scroll_count": "max",
    
    "scroll_length": "sum",
#     "scroll_speed": "mean",
    "scroll_time": "sum",
#     "sorted_scroll_time":"sum",
    
    "tap_count": "max",
#     "tap_interval": "mean",
    "tap_position_x": "max",
    "tap_position_y": "max",
    "ratio_tap_position_x": "max",
    "ratio_tap_position_y": "max",
    "view_position": "last",
    "event_response": "sum",
    "event_scroll": "sum" 
}

# グループ化して集計
df = data_clean .groupby("taskNum").agg(aggregation_functions).reset_index()

# 蓄積値を増加値に変更
df['s_choice_left'] = df['choice_left'].diff().fillna(df['choice_left'])
df['s_choice_right'] = df['choice_right'].diff().fillna(df['choice_right'])
df['s_tap_count'] = df['tap_count'].diff().fillna(df['tap_count'])
df['s_view_position'] = df['view_position'].diff().fillna(df['view_position'])

df = df.drop(columns=['choice_left','choice_right','tap_count','scroll_count','event_response','view_position','s_choice_right'	])

df['s_scroll_speed'] = df['scroll_length']/df['scroll_time']

df = df.replace([np.inf, -np.inf], np.nan)
df.fillna(0, inplace=True)  # NaNを0で埋める

csv_create_location = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/data０６２４.csv'
df.to_csv(csv_create_location, index = True)

df.head(151)

In [ ]:
# 正規化　勾配ブースティングは特徴量のスケールに対してロバストであるため必須ではない。

# モデル設計

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = set(df.columns) - {'isAnswerCorrect'}
target = 'isAnswerCorrect'

predictions = []

# 最初の行は予測をスキップし、適切な値を入れる
initial_prediction = 1
predictions.append(initial_prediction)

for n in range(1, len(df)):
    #n行目までのデータを訓練に、n+1行目をテストに
    train_data = df.iloc[:n]
    test_data = df.iloc[n:n+1]
    
    X_train =train_data[features]
    y_train = train_data[target]
    
    X_test = test_data[features]
    y_test = test_data[target]
    
    #クラスが２つ以上あるか
    if len(y_train.unique()) > 1:
        
        # NaN、Infinityを含むかチェックし、適切な値で埋める
        if np.any(np.isnan(X_test)) or np.any(np.isinf(X_test)):
           # print(X_test)
            X_test = X_test.replace([np.inf, -np.inf], 0)
            X_test = X_test.fillna(0)
            
        if np.any(np.isnan(X_train)) or np.any(np.isinf(X_train)):
            #  print(X_train) 
            X_train = X_train.replace([np.inf, -np.inf], 0)
            X_train = X_train.fillna(0)
            
       # LightGBM用データセットの作成
        train_dataset = lgb.Dataset(X_train, label=y_train)
        test_dataset = lgb.Dataset(X_test, label=y_test, reference=train_dataset)
        
        params = {
            'objective': 'binary',
            'metric': 'binary_error',
            'verbosity': -1
        }
        
        # モデルの作成と訓練
        model = lgb.train(params,train_dataset,num_boost_round=100,valid_sets=test_dataset )
        
        # 保存
        model.save_model(f'model_{n}.txt')

        # 予測
        y_pred = model.predict(X_test, num_iteration=model.best_iteration)
        y_pred = [1 if pred > 0.5 else 0 for pred in y_pred]

    else:
        
        #なかったら予測値は最頻値を使う
        y_pred = [y_train.mode()[0]]
        
    # 予測結果を保存
    predictions.append(y_pred[0])
    
# 予測結果をデータフレームに追加
df['predicted_isAnswerCorrect'] =  predictions

# 結果の表示
print(df)


In [ ]:
import os

# カレントディレクトリを表示
current_directory = os.getcwd()
print(f'Current Directory: {current_directory}')

actual = df[target].iloc[1:].astype(int)
predicted = df['predicted_isAnswerCorrect'].iloc[1:].astype(int)

accuracy = accuracy_score(actual, predicted)
print(f'Overwell accuracy: {accuracy}')

print(model.params)

# 特徴量の重要度出力
print(model.feature_importance())

# 特徴量の重要度をプロット
lgb.plot_importance(model)

# 図示
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(actual) + 1), actual, label='Actual', marker='o')
plt.plot(range(1, len(predicted) + 1), predicted, label='Predicted', marker='x')
plt.xlabel('taskNum')
plt.ylabel('isAnswerCorrect')
plt.title('Actual & Predicted isAnswerCorrect')
plt.legend()
plt.show()

# 分類結果のカウント
tp = np.sum((actual == 1) & (predicted == 1))
tn = np.sum((actual == 0) & (predicted == 0))
fp = np.sum((actual == 0) & (predicted == 1))
fn = np.sum((actual == 1) & (predicted == 0))

print(f'True Positives: {tp}')
print(f'True Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')

# グラフの描画
labels = ['TP (1 predicted as 1)', 'TN (0 predicted as 0)', 'FP (0 predicted as 1)', 'FN (1 predicted as 0)']
counts = [tp, tn, fp, fn]

plt.figure(figsize=(10, 6))
plt.bar(labels, counts, color=['green', 'blue', 'red', 'orange'])
plt.xlabel('Classification Results')
plt.ylabel('Count')
plt.title('Classification Results Count')
plt.show()

# 前回

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# モデルの訓練
model = LogisticRegression()
model.fit(X_train, y_train)

# 予測
y_pred = model.predict(X_test)

# 結果の表示
print(f　'Predicted isAnswerCorrect for taskNum == {n}: {y_pred}')


In [ ]:
# taskNumが1のデータ
train_data = df[df['taskNum'] == 1]
X_train = df.drop(columns=['isAnswerCorrect'])
y_train = df['isAnswerCorrect']

# taskNumが2のデータ
test_data = df[df['taskNum'] == 2]
X_test = test_data.drop(columns=['isAnswerCorrect'])
y_test = test_data['isAnswerCorrect']

# # 全てのカラムをfloatに変換
# data_clean = data_clean.astype(float)

# LightGBM用データセットの作成
train_dataset = lgb.Dataset(X_train, label=y_train)
test_dataset = lgb.Dataset(X_test, label=y_test, reference=train_dataset)

# モデルの設定
params = {
    'objective': 'binary',
    'metric':'auc',
}

# モデルの作成と訓練
model = lgb.train(params,train_dataset,num_boost_round=1000,valid_sets=test_dataset )

# 保存
model.save_model('model.txt')

# 予測
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

# AUC (Area Under the Curve) を計算する
fpr, tpr, thresholds = metrics.roc_curve(y_test,y_pred)
auc = metrics.auc(fpr, tpr)
print(auc)

# ROC曲線をプロット
plt.plot(fpr, tpr, label='ROC curve (area = %.2f)'%auc)
plt.legend()
plt.title('ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True)

# 特徴量の重要度出力
print(model.feature_importance())

# 特徴量の重要度をプロット
lgb.plot_importance(model)

In [ ]:
print(dataset['isAnswerCorrect'].value_counts())
Y = dataset['isAnswerCorrect']
Y = dataset['isAnswerCorrect'].replace({True: 1, False: 0})
print(Y.value_counts())

X = dataset.drop(columns=['isAnswerCorrect','sensor_id','data_time','event','Unnamed: 0'],axis = 1)
print(X.columns)

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=0,shuffle=True)
lgb_train=lgb.Dataset(X_train,Y_train)
lgb_eval=lgb.Dataset(X_test,Y_test,reference=lgb_train)

In [ ]:
from lightgbm import LGBMRegressor
params = {
    'objective': 'binary',
    'metrics':'binary_logloss',
#     'boosting_type':'gbdt',
#     'verbosity': -1,
#     'random_state': 42,
#     'learning_rate': 0.01,
#     'verbose_eval':50,
#     'early_stopping_rounds':100
}
print("タイプ：",type(lgb_train))
model = lgb.train(params,lgb_train,num_boost_round=1000,valid_sets=lgb_eval)
model.save_model('model.txt')
print(model.params)

In [ ]:
# AUC (Area Under the Curve) を計算する
Y_pred = model.predict(X_test, num_iteration=model.best_iteration)
fpr, tpr, thresholds = metrics.roc_curve(Y_test,Y_pred)
auc = metrics.auc(fpr, tpr)
print(auc)

# ROC曲線をプロット
plt.plot(fpr, tpr, label='ROC curve (area = %.2f)'%auc)
plt.legend()
plt.title('ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True)

# 特徴量の重要度出力
print(model.feature_importance())

# 特徴量の重要度をプロット
lgb.plot_importance(model)